In [1]:
## load all libraries

import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf,plot_pacf
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_percentage_error,mean_absolute_percentage_error
from statsmodels.tsa.seasonal import STL
import numpy as np
from pandas import Series, DataFrame
from scipy import stats
from statsmodels.tsa.stattools import adfuller
import statsmodels
from statsmodels.tsa.seasonal import seasonal_decompose
from pandas.plotting import register_matplotlib_converters
import pmdarima as pm
register_matplotlib_converters()
import warnings
import time
from numpy import array
from keras.models import Sequential
from keras.layers import LSTM
from keras.layers import Dense
from numpy import array
import keras_tuner as kt
import tensorflow as tf
print(tf.__version__)
from tensorflow import keras
import keras_tuner as kt
from sklearn.preprocessing import MinMaxScaler
from keras.layers import Bidirectional
from keras.models import Sequential
from keras.preprocessing.sequence import TimeseriesGenerator
from keras.layers import Bidirectional
from tensorflow.keras import initializers
import random as rn
np.random.seed(123)
rn.seed(123)
tf.random.set_seed(123)
tf.keras.utils.set_random_seed(123)
keras.utils.set_random_seed(123)
warnings.filterwarnings('ignore')
import os


2.10.0


In [65]:
# helpers
## stl analysis
def stl_analysis(ts, filename,category):
    results = seasonal_decompose(ts,period=7)
    results.plot();
    plt.savefig(category+'_stl_analysis/'+filename+'.png')  
    plt.close()
    
## scale data and get its characterstics
def data_scaling_and_getting_characteristics(ts):
    train_all = ts.iloc[:int(len(ts)*0.9)]
    train = ts.iloc[:int(len(ts)*0.7)]
    val = ts.iloc[int(len(ts)*0.7):int(len(ts)*0.9)]
    test = ts.iloc[int(len(ts)*0.9):int(len(ts)*0.9)+21]
    MIN= np.min(train_all)
    MAX= np.max(train_all)
    test_mean=np.mean(test)['unit_sales']
    
    scaler = MinMaxScaler()
    scaler.fit(train_all)
    scaled_all = scaler.transform(ts)
    scaled_train = scaler.transform(train)
    scaled_train_all = scaler.transform(train_all)
    scaled_val = scaler.transform(val)
    scaled_test = scaler.transform(test)

    return scaled_train_all,scaled_train,scaled_val,scaled_val,scaled_test,MIN,MAX,test_mean

def arima_forecasting(ts,scaled_val,scaled_test):
    ## Train ARIMA m=12 cause periodicity is one month (12 period per year)
    arima = pm.auto_arima(ts,
                         max_p=7,max_d=2,max_q=7,max_Q=4,max_P=4,max_D=2,
                         n_fits =200,
                         trace=False,m=7,trend=[1,1,0,0],
                         random=True, maxiter =200,max_order = None,alpha=0.01,n_jobs=-1,
                         information_criterion='oob',out_of_sample_size=len(scaled_val),random_state =10)

    arima_prediction=arima.predict(n_periods=len(scaled_test))
    arima_fitted, conf_int = arima.predict_in_sample(return_conf_int=True, alpha=0.05)
       
    return arima_prediction, arima_fitted


def lstm_forecasting(scaled_train_all,scaled_train,scaled_val,scaled_test,category, filename,hu,lr):
    # shaping data
    n_features = 1
    n_input = 9
    train_generator_all = TimeseriesGenerator(scaled_train_all, scaled_train_all, length=n_input, batch_size=2,shuffle=True)
    train_generator = TimeseriesGenerator(scaled_train, scaled_train, length=n_input, batch_size=2,shuffle=True)
    val_generator = TimeseriesGenerator(scaled_val, scaled_val, length=n_input, batch_size=2,shuffle=True)

    model = keras.Sequential()
    model.add(Bidirectional(LSTM(hu, activation='relu', return_sequences=True), input_shape=(n_input, n_features)))
    model.add(Bidirectional(LSTM(hu, activation='relu')))
    model.add(Dense(1))
    early_stopping_monitor = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=50,
        restore_best_weights=True
    )
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),  loss='mse')
    
    history = model.fit_generator(train_generator, validation_data=val_generator,epochs=200,shuffle=True, verbose=0,callbacks=[early_stopping_monitor])
    plt.plot(history.history['loss'],label='loss')
    plt.plot(history.history['val_loss'],label='val_loss')
    plt.savefig(category+'_learning_curve_1/'+filename+'.png')  
    plt.legend()
    plt.close()
    
    val_loss_per_epoch = history.history['val_loss']
    best_epoch = val_loss_per_epoch.index(min(val_loss_per_epoch)) + 1
    print('Best epoch: %d' % (best_epoch))
    
    last_train_batch = scaled_train_all[-n_input:]
    last_train_batch = last_train_batch.reshape((1, n_input, n_features))
    model.predict(last_train_batch)

    # predicting training and test data
    lstm_predictions = []

    first_eval_batch = scaled_train_all[-n_input:]
    current_batch = first_eval_batch.reshape((1, n_input, n_features))

    for i in range(len(scaled_test)):
        current_pred = model.predict(current_batch,verbose=0)[0]
        lstm_predictions.append(current_pred) 
        #current_batch = np.append(current_batch[:,1:,:],[[scaled_test[i]]],axis=1)
        current_batch = np.append(current_batch[:,1:,:],[[current_pred]],axis=1)

    lstm_fit = []
    first_fit_batch = scaled_train_all[:n_input]
    current_batch = first_fit_batch.reshape((1, n_input, n_features))

    for i in range(n_input):
        lstm_fit.append(0)

    for i in range(len(scaled_train_all)-n_input):
        current_fit = model.predict(current_batch,verbose=0)[0]
        lstm_fit.append(current_fit) 
        #current_batch = np.append(current_batch[:,1:,:],[[scaled_train_all[i]]],axis=1)
        current_batch = np.append(current_batch[:,1:,:],[[current_pred]],axis=1)
        
    return lstm_predictions,lstm_fit

def save_forecast_image(scaled_train_all,arima_fitted,arima_prediction,lstm_fit,lstm_predictions,category, filename,MIN,MAX):
    
    unscaled_train_all = scaled_train_all * int(MAX - MIN) + int(MIN)
    unscaled_test = scaled_test * int(MAX - MIN) + int(MIN)

    unscaled_arima_fitted = arima_fitted * int(MAX - MIN) + int(MIN)
    unscaled_arima_prediction = arima_prediction * int(MAX - MIN) + int(MIN)


    unscaled_lstm_fit=[x* int(MAX - MIN) for x in lstm_fit]
    unscaled_lstm_fit=[x+ int(MIN) for x in unscaled_lstm_fit]

    unscaled_lstm_predictions=[x* int(MAX - MIN) for x in lstm_predictions]
    unscaled_lstm_predictions=[x+ int(MIN) for x in unscaled_lstm_predictions]

    figure, ((ax1, ax2), (ax3, ax4) ) =plt.subplots(2, 2)
    figure.set_size_inches(20, 12)

    ax1.plot(unscaled_train_all,label = "Test")
    ax1.plot(unscaled_arima_fitted,label = "Prediction")
    ax1.legend()
    ax1.title.set_text('Train ARIMA')

    ax2.plot(unscaled_test,label = "Test")
    ax2.plot(unscaled_arima_prediction,label = "Prediction")
    ax2.legend()
    ax2.title.set_text('Test ARIMA')


    ax3.plot(unscaled_train_all,label = "Test")
    ax3.plot(unscaled_lstm_fit,label = "Prediction")
    ax3.legend()
    ax3.title.set_text('Train LSTM')

    ax4.plot(unscaled_test,label = "Test")
    ax4.plot(unscaled_lstm_predictions,label = "Prediction")
    ax4.legend()
    ax4.title.set_text('Test LSTM')
    
    plt.savefig(category+'_forecast/'+filename+'.png')  
    plt.close()

    return unscaled_test,unscaled_arima_prediction,unscaled_lstm_predictions

In [ ]:
# assign directory
directory = 'dataset/erratic_ts/'

total_stats = pd.DataFrame(columns=['hu','lr','arima_mse','arima_rmse','lstm_mse','lstm_rmse'])

# Gride searchou
for lr in [1e-2,1e-3,1e-4,1e-5]:    
    for hu in range(10,40,3):
        print("HU : "+str(hu)+" LR : "+str(lr))
        stats=0
        del stats
        stats = pd.DataFrame(columns=['arima_r2','arima_mse','arima_rmse','arima_mape','lstm_r2','lstm_mse','lstm_rmse','lstm_mape'])

        scaler = MinMaxScaler()
        # for each TS
        category ='erratic_hu_'+str(hu)+"_lr_"+str(lr)
        os.mkdir(category+'_forecast')
        os.mkdir(category+'_learning_curve_1')
        os.mkdir(category+'_stl_analysis')
        
        for filename in os.listdir(directory):
            print(filename)
            f = os.path.join(directory, filename)
            # checking if it is a file
            if os.path.isfile(f):
                # read TS
                df3 = pd.read_csv(f)
                df3.columns = ['date', 'unit_sales']
                df3.set_index('date',inplace=True)

                # stl decomposition 
                stl_analysis(df3['unit_sales'],filename,category) 

                # save cv2, adi, sd, mean, stationarity and get scaled data
                scaled_train_all,scaled_train,scaled_val,scaled_val,scaled_test,MIN,MAX,test_mean = data_scaling_and_getting_characteristics(df3)


                # perform arima forecasting 
                arima_prediction, arima_fitted= arima_forecasting(scaled_train_all,scaled_val,scaled_test)

                # perform LSTM forecasting 
                lstm_predictions,lstm_fit=lstm_forecasting(scaled_train_all,scaled_train,scaled_val,scaled_test,category, filename,hu,lr)



                # save graphs to an image
                unscaled_test,unscaled_arima_prediction,unscaled_lstm_predictions=save_forecast_image(scaled_train_all,arima_fitted,arima_prediction,lstm_fit,lstm_predictions,category, filename,MIN,MAX)


                lstm_r2 = r2_score(scaled_test,lstm_predictions)
                lstm_mse= mean_squared_error(scaled_test,lstm_predictions,squared=True)
                lstm_rmse = mean_squared_error(scaled_test,lstm_predictions, squared=False)
                lstm_mape = mean_absolute_percentage_error(scaled_test,lstm_predictions)

                arima_r2 = r2_score(scaled_test,arima_prediction)
                arima_mse= mean_squared_error(scaled_test,arima_prediction,squared=True)
                arima_rmse = mean_squared_error(scaled_test,arima_prediction, squared=False)
                arima_mape = mean_absolute_percentage_error(scaled_test,arima_prediction)

                values_to_add = {'test_mean': test_mean,'arima_r2': arima_r2,'arima_mse': arima_mse,'arima_rmse': arima_rmse,'arima_mape': arima_mape,'lstm_r2': lstm_r2,'lstm_mse': lstm_mse,'lstm_rmse': lstm_rmse,'lstm_mape': lstm_mape }
                row_to_add = pd.Series(values_to_add, name='x')
                stats = stats.append(row_to_add)

        print("arima mse : "+str(np.mean(stats['arima_mse'])))
        print("lstm mse : "+str(np.mean(stats['lstm_mse'])))

        print("arima rmse : "+str(np.mean(stats['arima_rmse'])))
        print("lstm rmse : "+str(np.mean(stats['lstm_rmse'])))
        
        total_values_to_add = {'hu':hu,'lr':lr,'arima_mse': np.mean(stats['arima_mse']),'arima_rmse': np.mean(stats['arima_rmse']),'lstm_mse': np.mean(stats['lstm_mse']),'lstm_rmse': np.mean(stats['lstm_rmse'])}
        total_row_to_add = pd.Series(total_values_to_add, name='x')
        total_stats = total_stats.append(total_row_to_add)


HU : 10 LR : 0.01
1_1038952.csv
Best epoch: 71
1/1 [==============================] - 0s 297ms/step
45_1004550.csv
Best epoch: 13
1/1 [==============================] - 0s 249ms/step
45_1004551.csv
Best epoch: 115
1/1 [==============================] - 0s 334ms/step
45_749729.csv
Best epoch: 109
1/1 [==============================] - 0s 330ms/step
46_364738.csv
Best epoch: 98
1/1 [==============================] - 0s 380ms/step
47_1004550.csv
Best epoch: 27
1/1 [==============================] - 0s 403ms/step
47_1004551.csv
Best epoch: 47
1/1 [==============================] - 0s 367ms/step
48_1004550.csv
Best epoch: 133
1/1 [==============================] - 0s 422ms/step
49_1004551.csv
Best epoch: 110
1/1 [==============================] - 0s 452ms/step
7_1004551.csv
Best epoch: 55
1/1 [==============================] - 0s 478ms/step
arima mse : 0.006869773689155973
lstm mse : 0.013802214359337112
arima rmse : 0.07596207405016063
lstm rmse : 0.10291770227872517
HU : 13 LR : 0.01
1_10

In [64]:

    print("miaw")